# COSC 419 — Jersey Number Recognition Pipeline
**Group 9 | SoccerNet Test Set | Final Accuracy: 87.12%**

---

## Before You Start — Manual File Requirements

Before running any cells, make sure you have the following files ready. These are **not** automatically downloaded and must be placed manually:

| File | Destination in Colab | Source |
|------|----------------------|--------|
| `vitpose-h.pth` (~2.5 GB) | Saved in Google Drive (script will copy it) | ViTPose HuggingFace / course materials |
| `parseq_epoch=3-step=95-val_accuracy=98.7903-val_NED=99.3952.ckpt` | `models/` folder in repo | Course-provided or trained model |
| `test.zip` | `MyDrive/COSC 419/data/jersey-2023/jersey-2023/test.zip` | SoccerNet download |
| `evaluate.py` | `/content/jersey-number-pipeline/evaluate.py` | Provided in this repo |

**Google Drive backup folder expected at:** `MyDrive/jersey-number-pipeline-backup/`

This folder should contain `SoccerNetResults/` from a previous run (features, legibility, pose results). If starting fresh, all pipeline stages must be enabled in Step 14.

---

## Pipeline Overview
```
SoccerNet Test Images
        ↓
1. Soccer Ball Detection       (jnp env)
2. Feature Generation / ReID   (centroids env)  ← ~84 min on T4
3. Gaussian Outlier Filtering  (jnp env)
4. Legibility Classification   (jnp env)         ← ~48 min on T4
5. Pose Estimation             (vitpose env)      ← ~2 hrs on T4 / ~4 hrs on A100
6. Crop Generation             (jnp env)          ← ~2 min
7. Jersey Number Recognition   (parseq2 env)      ← ~22 min
8. Combine + Evaluate          (jnp env)
```
Steps 1–5 results are saved to Drive. On subsequent runs, restore them and start from Step 6.


---
## Part 1: Environment Setup
---

### Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
backup = '/content/drive/MyDrive/jersey-number-pipeline-backup'
print("Drive backup contents:", os.listdir(backup) if os.path.exists(backup) else "Backup not found")

### Step 2 — Clone Repo & Install micromamba

In [ ]:
!git clone https://github.com/Group-9-Cosc-419/Cosc_419-JerseyNumber-Recognition.git /content/jersey-number-pipeline
%cd /content/jersey-number-pipeline

In [ ]:
%cd /content
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
!chmod +x /content/bin/micromamba
!/content/bin/micromamba --version

### Step 3 — Create `conda` shim (pipeline uses conda internally)

In [ ]:
%%bash
mkdir -p /content/fake_conda
cat > /content/fake_conda/conda << 'EOF'
#!/bin/bash
exec /content/bin/micromamba "$@"
EOF
chmod +x /content/fake_conda/conda
echo "Done"

### Step 4 — Setup `jnp` environment (main pipeline env)

In [ ]:
!/content/bin/micromamba create -y -n jnp python=3.9
!/content/bin/micromamba install -y -n jnp -c conda-forge numpy==1.23.5 pandas scipy
!/content/bin/micromamba run -n jnp pip install \
    opencv-python tqdm pyyaml matplotlib scikit-learn pillow gdown timm einops lmdb shapely SoccerNet

In [ ]:
# Install CUDA-enabled PyTorch for jnp
!/content/bin/micromamba run -n jnp pip install torch torchvision \
    --index-url https://download.pytorch.org/whl/cu124 --force-reinstall
!/content/bin/micromamba run -n jnp python -c "import torch; print('jnp:', torch.__version__, '| CUDA:', torch.cuda.is_available())"

In [ ]:
# Fix scipy compatibility
!/content/bin/micromamba run -n jnp pip install scipy --force-reinstall

# Fix torchvision numpy compatibility (numpy 2.x broke np.array copy API)
path = "/root/.local/share/mamba/envs/jnp/lib/python3.9/site-packages/torchvision/transforms/functional.py"
with open(path, "r") as f:
    code = f.read()
patched = code.replace(
    "img = torch.from_numpy(np.array(pic, mode_to_nptype.get(pic.mode, np.uint8), copy=True))",
    "img = torch.from_numpy(np.asarray(pic, dtype=mode_to_nptype.get(pic.mode, np.uint8)).copy())"
)
with open(path, "w") as f:
    f.write(patched)
print("jnp patches applied!")

### Step 5 — Setup `centroids` environment (ReID / feature generation)

In [ ]:
!/content/bin/micromamba create -y -n centroids python=3.9
!/content/bin/micromamba install -y -n centroids -c conda-forge numpy==1.23.5 pandas scipy
!/content/bin/micromamba run -n centroids pip install \
    opencv-python tqdm scikit-learn pillow pyyaml yacs einops

In [ ]:
# Install CUDA-enabled PyTorch for centroids
!/content/bin/micromamba run -n centroids pip install torch torchvision \
    --index-url https://download.pytorch.org/whl/cu124 --force-reinstall
!/content/bin/micromamba run -n centroids python -c "import torch; print('centroids:', torch.__version__, '| CUDA:', torch.cuda.is_available())"

In [ ]:
# Install pytorch-lightning stack with pinned compatible versions
!/content/bin/micromamba run -n centroids pip install pytorch-lightning==1.4.9 torchmetrics==0.6.0
!/content/bin/micromamba install -y -n centroids -c conda-forge "setuptools<81" packaging tensorboard
!/content/bin/micromamba run -n centroids pip install "numpy<2.0" --force-reinstall

In [ ]:
# Patch 1: Fix PyTorch 2.6 weights_only default breaking checkpoint loading
path = "/root/.local/share/mamba/envs/centroids/lib/python3.9/site-packages/pytorch_lightning/utilities/cloud_io.py"
with open(path, "r") as f:
    code = f.read()
with open(path, "w") as f:
    f.write(code.replace(
        "return torch.load(f, map_location=map_location)",
        "return torch.load(f, map_location=map_location, weights_only=False)"
    ))

# Patch 2: Fix torchvision numpy compatibility
path = "/root/.local/share/mamba/envs/centroids/lib/python3.9/site-packages/torchvision/transforms/functional.py"
with open(path, "r") as f:
    code = f.read()
with open(path, "w") as f:
    f.write(code.replace(
        "img = torch.from_numpy(np.array(pic, mode_to_nptype.get(pic.mode, np.uint8), copy=True))",
        "img = torch.from_numpy(np.asarray(pic, dtype=mode_to_nptype.get(pic.mode, np.uint8)).copy())"
    ))

print("centroids patches applied!")

### Step 6 — Setup `centroids-reid` repo & download weights

In [ ]:
%cd /content/jersey-number-pipeline/reid
!rm -rf centroids-reid
!git clone https://github.com/mikwieczorek/centroids-reid.git centroids-reid
!mkdir -p /content/jersey-number-pipeline/reid/centroids-reid/models

In [ ]:
# Download Market1501 ReID weights
!pip install gdown -q
!gdown --folder https://drive.google.com/drive/folders/1NWD2Q0JGasGm9HTcOy4ZqsIqK4-IfknK \
    -O /content/jersey-number-pipeline/reid/centroids-reid/models/
!cp /content/jersey-number-pipeline/reid/centroids-reid/models/reid_weights/market1501_resnet50_256_128_epoch_120.ckpt \
    /content/jersey-number-pipeline/reid/centroids-reid/models/market1501_resnet50_256_128_epoch_120.ckpt
!ls -lh /content/jersey-number-pipeline/reid/centroids-reid/models/

In [ ]:
# Patch bases.py: PyTorch Lightning >=1.6 made self.hparams read-only
path = "/content/jersey-number-pipeline/reid/centroids-reid/modelling/bases.py"
with open(path, "r") as f:
    code = f.read()
with open(path, "w") as f:
    f.write(code.replace(
        "self.hparams = AttributeDict(hparams)",
        "self._set_hparams(AttributeDict(hparams))"
    ))
print("bases.py patched!")

### Step 7 — Setup `vitpose` environment (pose estimation)

In [ ]:
!/content/bin/micromamba create -y -n vitpose python=3.8
!/content/bin/micromamba run -n vitpose pip install torch==2.1.0 torchvision \
    --index-url https://download.pytorch.org/whl/cu118 --force-reinstall
!/content/bin/micromamba run -n vitpose python -c "import torch; print('vitpose:', torch.__version__, '| CUDA:', torch.cuda.is_available())"

In [ ]:
# Install mmcv (lite version — mmcv-full compilation is too fragile)
!/content/bin/micromamba run -n vitpose pip install mmcv==1.5.0
!/content/bin/micromamba run -n vitpose pip install timm matplotlib

# Clone ViTPose repo and install as package (registers ViT backbone)
!git clone https://github.com/ViTAE-Transformer/ViTPose.git /content/jersey-number-pipeline/pose/ViTPose_repo
!/content/bin/micromamba run -n vitpose pip install -e /content/jersey-number-pipeline/pose/ViTPose_repo/ --no-deps
!/content/bin/micromamba run -n vitpose pip install mmdet xtcocotools
!/content/bin/micromamba run -n vitpose pip install chumpy json_tricks munkres

In [ ]:
# Copy ViTPose configs into expected location
!cp -r /content/jersey-number-pipeline/pose/ViTPose_repo/configs \
    /content/jersey-number-pipeline/pose/ViTPose/

# Patch checkpoint.py to load weights directly to CUDA
path = "/root/.local/share/mamba/envs/vitpose/lib/python3.8/site-packages/mmcv/runner/checkpoint.py"
with open(path, "r") as f:
    code = f.read()
with open(path, "w") as f:
    f.write(code.replace(
        "checkpoint = torch.load(filename, map_location=map_location)",
        'checkpoint = torch.load(filename, map_location="cuda:0")'
    ))

print("vitpose patches applied!")
!/content/bin/micromamba run -n vitpose python3 -c \
    "from mmpose.apis import inference_top_down_pose_model, init_pose_model; print('vitpose mmpose OK')"

### Step 8 — Setup `parseq2` environment (jersey number recognition / STR)

In [ ]:
!/content/bin/micromamba create -y -n parseq2 python=3.9
!/content/bin/micromamba run -n parseq2 pip install "setuptools<70"
!/content/bin/micromamba run -n parseq2 pip install torch==2.1.0 torchvision==0.16.0 \
    --index-url https://download.pytorch.org/whl/cu124
!/content/bin/micromamba run -n parseq2 pip install \
    pytorch-lightning==1.8.6 torchmetrics==0.11.4 \
    tqdm opencv-python pillow numpy timm lmdb
!/content/bin/micromamba run -n parseq2 pip install \
    -e /content/jersey-number-pipeline/str/parseq/ --no-deps

In [ ]:
# Patch lightning_lite cloud_io.py (weights_only fix for PyTorch 2.x)
path = "/root/.local/share/mamba/envs/parseq2/lib/python3.9/site-packages/lightning_lite/utilities/cloud_io.py"
with open(path, "r") as f:
    code = f.read()
with open(path, "w") as f:
    f.write(code.replace(
        "return torch.load(f, map_location=map_location)",
        "return torch.load(f, map_location=map_location, weights_only=False)"
    ))
print("parseq2 patched!")

### Step 9 — Setup SAM optimizer & download legibility model

In [ ]:
%cd /content/jersey-number-pipeline
!rm -rf sam
!git clone https://github.com/davda54/sam sam

In [ ]:
# Download legibility classifier weights
%cd /content/jersey-number-pipeline
!mkdir -p models
!/content/bin/micromamba run -n jnp gdown \
    "https://drive.google.com/uc?id=18HAuZbge3z8TSfRiX_FzsnKgiBs-RRNw" \
    -O /content/jersey-number-pipeline/models/legibility_resnet34_soccer_20240215.pth
!ls -lh models/

### Step 10 — Place parseq model weights

> ⚠️ **Manual step required!** Copy your trained parseq checkpoint into the `models/` folder.
>
> Expected filename: `parseq_epoch=3-step=95-val_accuracy=98.7903-val_NED=99.3952.ckpt`
>
> Run the cell below to copy from Drive if you saved it there, or upload it manually.

In [ ]:
import shutil, os, glob

# Try to find and copy parseq weights from Drive
parseq_files = glob.glob('/content/drive/MyDrive/**/*.ckpt', recursive=True)
parseq_files = [f for f in parseq_files if 'parseq' in f.lower()]

if parseq_files:
    dest = '/content/jersey-number-pipeline/models/' + os.path.basename(parseq_files[0])
    shutil.copy(parseq_files[0], dest)
    print(f"Copied: {parseq_files[0]} → {dest}")
else:
    print("No parseq .ckpt found in Drive. Please upload manually to:")
    print("/content/jersey-number-pipeline/models/")

!ls -lh /content/jersey-number-pipeline/models/

### Step 11 — Patch `main.py` (replace `conda run` with `micromamba`)

In [ ]:
with open('/content/jersey-number-pipeline/main.py', 'r') as f:
    code = f.read()

# Replace all conda run with micromamba
code = code.replace('conda run -n', '/content/bin/micromamba run -n')

# Ensure pose command includes --device cuda:0
if '--device cuda:0' not in code:
    code = code.replace(
        '--out-json {output_json}"',
        '--out-json {output_json} --device cuda:0"'
    )

with open('/content/jersey-number-pipeline/main.py', 'w') as f:
    f.write(code)

print("main.py patched!")
!grep -n "micromamba" /content/jersey-number-pipeline/main.py | head -5

### Step 12 — Patch `pose.py` (add resume/checkpoint support)

In [ ]:
# Adds ability to resume pose detection if interrupted
# Saves progress every 100 images, skips already-processed images on restart
path = '/content/jersey-number-pipeline/pose.py'
with open(path, 'r') as f:
    code = f.read()

old = '''    results = []

    # process each image
    for i in range(len(img_keys)):'''

new = '''    results = []

    # Load existing results to resume if interrupted
    out_json = args.out_json
    processed_ids = set()
    if os.path.exists(out_json):
        with open(out_json, 'r') as fp:
            existing = json.load(fp)
            results = existing.get("pose_results", [])
            processed_ids = {r["id"] for r in results}
        print(f"Resuming: {len(processed_ids)} images already processed")

    # process each image
    for i in range(len(img_keys)):'''

old2 = '''        results.append(
            {"img_name": image['file_name'], "id": image_id, "keypoints": pose_results[0]['keypoints'].tolist()})'''

new2 = '''        # Skip already processed images
        if image_id in processed_ids:
            continue

        results.append(
            {"img_name": image['file_name'], "id": image_id, "keypoints": pose_results[0]['keypoints'].tolist()})

        # Checkpoint every 100 images
        if len(results) % 100 == 0:
            with open(out_json, 'w') as fp:
                json.dump({"pose_results": results}, fp)
            print(f"Checkpoint saved: {len(results)} images processed")'''

code = code.replace(old, new).replace(old2, new2)
with open(path, 'w') as f:
    f.write(code)
print("pose.py patched with resume support!")

### Step 13 — Restore saved outputs from Drive

> If running for the first time, skip this and enable all pipeline stages in Step 15.

In [ ]:
import shutil, os

backup = '/content/drive/MyDrive/jersey-number-pipeline-backup'
os.makedirs('/content/jersey-number-pipeline/out', exist_ok=True)

if os.path.exists(f'{backup}/SoccerNetResults'):
    shutil.copytree(
        f'{backup}/SoccerNetResults',
        '/content/jersey-number-pipeline/out/SoccerNetResults',
        dirs_exist_ok=True
    )
    print("SoccerNetResults restored!")
    !ls /content/jersey-number-pipeline/out/SoccerNetResults/
else:
    print("No backup found — will run full pipeline from scratch.")

### Step 14 — Place ViTPose weights

> ⚠️ **Manual step!** The `vitpose-h.pth` file (~2.5 GB) must be in Drive.
> Expected at: `MyDrive/jersey-number-pipeline-backup/ViTPose/checkpoints/vitpose-h.pth`

In [ ]:
import shutil, os

!mkdir -p /content/jersey-number-pipeline/pose/ViTPose/checkpoints

candidates = [
    '/content/drive/MyDrive/jersey-number-pipeline-backup/ViTPose/vitpose-h.pth',
    '/content/drive/MyDrive/jersey-number-pipeline-backup/ViTPose/checkpoints/vitpose-h.pth',
]
dest = '/content/jersey-number-pipeline/pose/ViTPose/checkpoints/vitpose-h.pth'

copied = False
for src in candidates:
    if os.path.exists(src):
        shutil.copy(src, dest)
        print(f"ViTPose weights copied from: {src}")
        copied = True
        break

if not copied:
    print("⚠️  vitpose-h.pth not found in Drive. Upload it manually to:")
    print(dest)

!ls -lh /content/jersey-number-pipeline/pose/ViTPose/checkpoints/

### Step 15 — Unzip SoccerNet test data

In [ ]:
!mkdir -p /content/jersey-number-pipeline/data/SoccerNet
!unzip -o "/content/drive/MyDrive/COSC 419/data/jersey-2023/jersey-2023/test.zip" \
    -d /content/jersey-number-pipeline/data/SoccerNet/
!ls /content/jersey-number-pipeline/data/SoccerNet/test/images | head -5

### Step 16 — Verify all environments and files

In [ ]:
print("=== GPU ===")
!nvidia-smi | grep -E 'T4|A100|V100|Blackwell|RTX|Name'

print("\n=== CUDA checks per env ===")
!/content/bin/micromamba run -n centroids python3 -c "import torch; print('centroids:', torch.__version__, torch.cuda.is_available())"
!/content/bin/micromamba run -n jnp       python3 -c "import torch; print('jnp:      ', torch.__version__, torch.cuda.is_available())"
!/content/bin/micromamba run -n vitpose   python3 -c "import torch; print('vitpose:  ', torch.__version__, torch.cuda.is_available())"
!/content/bin/micromamba run -n parseq2   python3 -c "import torch; print('parseq2:  ', torch.__version__, torch.cuda.is_available())"

print("\n=== Model weights ===")
!ls -lh /content/jersey-number-pipeline/models/
!ls -lh /content/jersey-number-pipeline/pose/ViTPose/checkpoints/

print("\n=== Restored outputs ===")
!ls /content/jersey-number-pipeline/out/SoccerNetResults/ 2>/dev/null || echo "No outputs yet"

---
## Part 2: Run the Pipeline
---

### Step 17 — Run Pose Estimation

> **⏱ Runtime: ~2 hrs on T4, ~4 hrs on A100 (105,170 images)**
>
> This step runs directly (not through `run_pipeline.py`) to ensure correct GPU usage.
> If interrupted, it will automatically resume from where it left off thanks to the patch in Step 12.
>
> ⚠️ **Note on GPU type:** This requires a T4 or A100 (sm_80/sm_86/sm_90). The RTX PRO 6000 Blackwell (sm_120) is incompatible with PyTorch 2.1. If you get a CUDA kernel error, switch to T4 via Runtime → Change runtime type.

In [ ]:
%cd /content/jersey-number-pipeline
!/content/bin/micromamba run -n vitpose python3 pose.py \
    pose/ViTPose/configs/body/2d_kpt_sview_rgb_img/topdown_heatmap/coco/ViTPose_huge_coco_256x192.py \
    pose/ViTPose/checkpoints/vitpose-h.pth \
    --img-root / \
    --json-file /content/jersey-number-pipeline/out/SoccerNetResults/pose_input.json \
    --out-json /content/jersey-number-pipeline/out/SoccerNetResults/pose_results.json \
    --device cuda:0

In [ ]:
# Save pose results to Drive immediately after completion
import shutil, os
os.makedirs('/content/drive/MyDrive/jersey-number-pipeline-backup/SoccerNetResults', exist_ok=True)
shutil.copy(
    '/content/jersey-number-pipeline/out/SoccerNetResults/pose_results.json',
    '/content/drive/MyDrive/jersey-number-pipeline-backup/SoccerNetResults/pose_results.json'
)
print("pose_results.json saved to Drive!")

### Step 18 — Run Remaining Pipeline (Crops → STR → Combine → Evaluate)

> **Set flags to `False` for any stages whose outputs are already in `out/SoccerNetResults/`**

In [ ]:
script = '''
import sys
sys.path.insert(0, '/content/jersey-number-pipeline')
sys.path.insert(0, '/content/jersey-number-pipeline/reid/centroids-reid')
import main

class Args:
    dataset = 'SoccerNet'
    part = 'test'
    pipeline = {
        'soccer_ball_filter': False,  # Stage 1 - set True if running from scratch
        'feat': False,                # Stage 2 - ~84 min, set True if running from scratch
        'filter': False,              # Stage 3 - set True if running from scratch
        'legible': False,             # Stage 4 - ~48 min, set True if running from scratch
        'legible_eval': False,
        'pose': False,                # Stage 5 - done in Step 17 above
        'crops': True,                # Stage 6 - ~2 min
        'str': True,                  # Stage 7 - ~22 min
        'combine': True,              # Stage 8
        'eval': True                  # Stage 9
    }

args = Args()
main.soccer_net_pipeline(args)
'''

with open('/content/run_pipeline.py', 'w') as f:
    f.write(script)
print("run_pipeline.py written!")

In [ ]:
%cd /content/jersey-number-pipeline
!PYTHONPATH="/content/jersey-number-pipeline:/content/jersey-number-pipeline/reid/centroids-reid:$PYTHONPATH" \
    PATH="/content/fake_conda:$PATH" \
    /content/bin/micromamba run -n jnp python3 /content/run_pipeline.py

---
## Part 3: Evaluation
---

### Step 19 — Evaluate against ground truth

> Uses the `evaluate.py` script provided by the course instructor.
>
> ⚠️ **Manual step:** Make sure `evaluate.py` is placed at `/content/jersey-number-pipeline/evaluate.py`

In [ ]:
# Find output predictions and GT file
!find /content/jersey-number-pipeline/out -name "final_results.json"
!find /content/jersey-number-pipeline/data -name "test_gt.json"

In [ ]:
!python /content/jersey-number-pipeline/evaluate.py \
    --pred /content/jersey-number-pipeline/out/SoccerNetResults/final_results.json \
    --gt /content/jersey-number-pipeline/data/SoccerNet/test/test_gt.json

---
## Part 4: Save Results to Drive
---

### Step 20 — Backup outputs to Drive (skips existing files)

In [ ]:
import subprocess
result = subprocess.run([
    'rsync', '-av', '--ignore-existing',
    '/content/jersey-number-pipeline/out/SoccerNetResults/',
    '/content/drive/MyDrive/jersey-number-pipeline-backup/SoccerNetResults/'
], capture_output=True, text=True)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
print("Backup complete!")